### TODO: Miles to km (?)

### TODO: Provide a detailed description of the trip dataset such that there are no pending questions.

### TODO: Evaluate Aggregation Logic especially in regards to Spatial Analysis and Prediction tasks

### TODO: Make Markdown Text look good

### TODO: Add column description from website here as Markdown

### TODO: Add Outlier Analysis

In [ ]:
import pandas as pd
import numpy as np
import h3

## What happened before uploading the CSV:
- Filtering the data for:
    - Pickup/Dropoff Census Tract is not null
    - Trip Seconds/Miles is not 0
    - Removing unneccasssary columns: Fare, Tips, Tolls, Extras, Payment Type, Pickup/Dropoff Centroid Location
- resulting data with 16 columns and 6.041.177 rows

In [ ]:
taxi_data = pd.read_csv("../data/Taxi_Trips.csv")
taxi_data

In [ ]:
taxi_data.info()
taxi_data.describe()

## 1. Change Data types from string to numeric

In [ ]:
# Columns to fix data type
cols_to_fix = [
    'Pickup Centroid Longitude', 'Pickup Centroid Latitude',
    'Dropoff Centroid Longitude', 'Dropoff Centroid Latitude',
    'Trip Total', 'Trip Miles'
]

for col in cols_to_fix:
    
    if col == 'Trip Total':
        # Remove the dollar sign and commas from the Trip Total column
        taxi_data[col] = taxi_data[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
    elif col == 'Trip Miles' or col in ['Pickup Centroid Longitude', 'Pickup Centroid Latitude', 'Dropoff Centroid Longitude', 'Dropoff Centroid Latitude']:
        # Replace the comma with a standard decimal point
        taxi_data[col] = taxi_data[col].astype(str).str.replace(',', '.', regex=False)

    # Convert to numeric
    taxi_data[col] = pd.to_numeric(taxi_data[col], errors='coerce')

# 2. Null Value Analysis

In [ ]:
# Find Null Values and add percentage of NaN values for each column
is_na_df = taxi_data.isna().sum()
is_na_df = pd.DataFrame(is_na_df, columns=['NaN Count'])
is_na_df['Total Count'] = len(taxi_data)
is_na_df['NaN Percentage'] = (is_na_df['NaN Count'] / is_na_df['Total Count']) * 100

# Check for any NaN values in all columns
print("NaN values before conversion:")
print(is_na_df)

## 3. Add Column with H3 index

In [ ]:
# Generate H3 indices for pickup and dropoff locations, we use resolution 8 as a starting point
taxi_data['h3_index_pickup_7'] = [
    # Check if lat and lng are not NaN before converting to H3 index, otherwise return None
    h3.latlng_to_cell(lat, lng, 7) if (lat == lat and lng == lng) else None
    for lat, lng in zip(
        taxi_data['Pickup Centroid Latitude'], 
        taxi_data['Pickup Centroid Longitude']
    )
]
taxi_data['h3_index_dropoff_7'] = [
    # Check if lat and lng are not NaN before converting to H3 index, otherwise return None
    h3.latlng_to_cell(lat, lng, 7) if (lat == lat and lng == lng) else None
    for lat, lng in zip(
        taxi_data['Dropoff Centroid Latitude'], 
        taxi_data['Dropoff Centroid Longitude']
    )
]

# Generate H3 indices for pickup and dropoff locations, we use resolution 8 as a starting point
taxi_data['h3_index_pickup_8'] = [
    # Check if lat and lng are not NaN before converting to H3 index, otherwise return None
    h3.latlng_to_cell(lat, lng, 8) if (lat == lat and lng == lng) else None
    for lat, lng in zip(
        taxi_data['Pickup Centroid Latitude'], 
        taxi_data['Pickup Centroid Longitude']
    )
]
taxi_data['h3_index_dropoff_8'] = [
    # Check if lat and lng are not NaN before converting to H3 index, otherwise return None
    h3.latlng_to_cell(lat, lng, 8) if (lat == lat and lng == lng) else None
    for lat, lng in zip(
        taxi_data['Dropoff Centroid Latitude'], 
        taxi_data['Dropoff Centroid Longitude']
    )
]

In [ ]:
# Check for any NaN values in the new H3 index columns
# TODO: Add percentage of NaN values in the H3 index columns
print(taxi_data[['h3_index_pickup_7', 
                 'h3_index_dropoff_7',
                 'h3_index_pickup_8',
                 'h3_index_dropoff_8',
                 'Pickup Census Tract', 
                 'Dropoff Census Tract', 
                 'Pickup Community Area', 
                 'Dropoff Community Area']].isna().sum())

columns_to_drop = [
    #'Pickup Centroid Longitude', 'Pickup Centroid Latitude',
    'Dropoff Centroid Longitude', 'Dropoff Centroid Latitude',
    # 'Pickup Census Tract', 'Dropoff Census Tract', # Maybe use this for spatial analysis
    #'Pickup Community Area', 'Dropoff Community Area',
]

# Drop rows where either pickup or dropoff H3 index is NaN, as these rows cannot be used for spatial analysis 
# and drop columns that are not needed for the analysis
# TODO: Consider more sophisticated methods for handling missing values in the coordinate columns, e.g. by imputing with mean/median or using a separate category for missing values, instead of dropping rows with NaN values in the H3 index columns.
taxi_data_processed = taxi_data.drop(columns=columns_to_drop
                                     ).dropna(subset=['h3_index_pickup_8', 'h3_index_dropoff_8', 
                                                      'h3_index_pickup_7', 'h3_index_dropoff_7',
                                                      'Taxi ID', 'Trip Total', 'Trip Miles'])

In [ ]:
taxi_data_processed.to_parquet(
    "../data/processed/taxi_data_processed.parquet"
)